In [1]:
import os
import re
import json
import torch
import pandas as pd
import scanpy as sc
import math
import logging
import importlib
from pathlib import Path
import warnings
import numpy as np
import traceback
import scipy.sparse as sp
import random
from scipy.special import softmax
from concurrent.futures import ProcessPoolExecutor, as_completed
from typing import Tuple, Set, Optional, List, Iterable, Union, Dict, Callable, Any
from anndata import AnnData
import pybedtools
import argparse
import pickle
from sklearn.decomposition import TruncatedSVD

import sys
PROJECT_DIR = "/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER"
SRC_DIR = str(Path(PROJECT_DIR) / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from multiomic_transformer.utils.peaks import find_genes_near_peaks, format_peaks
import multiomic_transformer.utils.data_formatter as data_formatter

random.seed(1337)
np.random.seed(1337)
torch.manual_seed(1337)

# Training Data Preparation and Caching

The model uses cached per-chromosome data to help speed up training. Here, we will go through the process of building the necessary tensors and reference files to train the gene prediction model.

## Set up the TrainingDataFormatter

The training data formatter ensures that the correct data files and output directories exist and helps to keep everything standardized.

In [2]:
importlib.reload(data_formatter)


<module 'multiomic_transformer.utils.data_formatter' from '/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/src/multiomic_transformer/utils/data_formatter.py'>

In [3]:
# Path to the project directory (same as Git repository root)
project_dir = Path(PROJECT_DIR)

# Path to the training output directory. Used to store the preprocessing config
output_dir = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/experiments")

# Name of the dataset / experiment to run
dataset_name = "mESC_E7.5_rep1_muon_preprocessing_filter_tfs"

# Organism code for the dataset. Supports either "mm10" or "hg38"
organism_code = "mm10"

# List of samples in the training datset. 
# Each of these should have its own subdirectory in the processed data directory
sample_names = ["E7.5_rep1"]

# List of chromosomes. Used to split the data by chromsome for caching and training.
# Should be in the format "chr1", "chr2", etc. and should match the chromosome names in the processed data files.
chrom_list = [f"chr{i}" for i in range(1, 5)]

tdf = data_formatter.TrainingDataFormatter(
    project_dir=project_dir,
    dataset_name=dataset_name,
    organism_code=organism_code,
    sample_names=sample_names,
    chrom_list=chrom_list,
    output_dir=output_dir / dataset_name,
)

with open(tdf.output_dir / "file_paths.json", "r") as f:
    file_paths = json.load(f)
    


  - Found existing genome file: /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/genome_data/reference_genome/mm10/mm10.fa.gz
  - mm10.fa.gz is already BGZF; skipping transcode
  - Index already exists: mm10.fa.gz.fai, mm10.fa.gz.gzi
Genome ready: /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/genome_data/reference_genome/mm10/mm10.fa.gz
Found existing gene_info: /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/genome_data/genome_annotation/mm10
Found existing GTF: /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/genome_data/genome_annotation/mm10/Mus_musculus.GRCm39.115.gtf.gz


Map sizes: {'n_official': 143320, 'n_ens2sym': 77920, 'n_entrez2sym': 112254, 'n_alias2sym': 72569}


## Sample-Level Data Preparation

We first load the pseudobulk data from the Muon QC and preprocssing pipeline. The data files are loaded and the peak and gene names are standardized.

We load the pseudobulk data for the sample, which also creates a list of gene names. The `split_genes_into_tfs_and_tgs` function checks which of the genes in the gene name list overlap with an organism-specific [file of TF names from CIS-BP](https://mitra.stanford.edu/kundaje/marinovg/oak/various/motifs/CIS-BP/TF_Information_all_motifs.txt) to create `tfs` and `tgs` attributes.


In [4]:
sample_name = sample_names[0]

pseudobulk_rna_df = tdf.load_pseudobulk_rna_df(sample_name)

print(f"Number of genes: {len(tdf.genes):,}")

# Next the genes are classified as TFs or TGs by checking which genes in the data are in the reference TF list. 
tdf.tfs, tdf.tgs = tdf.split_genes_into_tfs_and_tgs(tdf.genes)
print(f"  - TFs: {len(tdf.tfs):,} (First 3: {tdf.tfs[:3]})")
print(f"  - TGs: {len(tdf.tgs):,} (First 3: {tdf.tgs[:3]})")

Number of genes: 18,862
  - TFs: 1,223 (First 3: ['2010315B03RIK', '2310033P09RIK', '2610008E11RIK'])
  - TGs: 17,627 (First 3: ['0610005C13RIK', '0610009E02RIK', '0610009L18RIK'])


The pseudobulk ATAC data is loaded, which also creates a list of peak names

In [5]:
pseudobulk_atac_df = tdf.load_pseudobulk_atac_df(sample_name)

print(f"Number of peaks: {len(tdf.peaks):,} (First 3: {tdf.peaks[:3]})")

Number of peaks: 199,885 (First 3: ['chr1:3035460-3036350', 'chr1:3044677-3045614', 'chr1:3062482-3063387'])


A BED file of the peak locations is created using `create_peak_bed_file` and the peak locations are stored as a dataframe in `tdf.peak_locs_df`.

In [6]:
tdf.create_peak_bed_file(pseudobulk_atac_df, sample_name)
display(tdf.peak_locs_df.head())

,chrom,start,end,strand,peak_id
0,chr1,3035460,3036350,.,chr1:3035460-3036350
1,chr1,3044677,3045614,.,chr1:3044677-3045614
2,chr1,3062482,3063387,.,chr1:3062482-3063387
3,chr1,3072145,3073018,.,chr1:3072145-3073018
4,chr1,3191389,3192286,.,chr1:3191389-3192286


### Calculate peak to gene distance

Next, we will use the peak bed file created when loading the peak pseudobulk dataset to calculate the distance between each peak and each gene within 1 Mb. This will create a bias tensor to help the model prioritize genes that are closer to peaks when training the peak-TG cross-attention portion of the model.

In [7]:
tdf.create_peak_to_tg_distance_file(
    sample_name=sample_name,
    max_peak_distance=1e6,
    distance_factor_scale=25000,
    force_recalculate=False,
    filter_to_nearest_gene=False,
)

Peak to TSS distance file already exists for sample E7.5_rep1 at /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/processed/mESC_E7.5_rep1_muon_preprocessing_filter_tfs/E7.5_rep1/peak_to_gene_dist.parquet. Skipping recalculation.


,peak_chr,peak_start,peak_end,peak_id,gene_chr,gene_start,gene_end,target_id,TSS_dist,TSS_dist_score
0,chr9,55148902,55149745,chr9:55148902-55149745,chr9,55149745,55149746,NRG4,0,1.000000e+00
1,chr7,27251562,27252441,chr7:27251562-27252441,chr7,27252441,27252442,PLD3,0,1.000000e+00
2,chr14,55918361,55919232,chr14:55918361-55919232,chr14,55919232,55919233,TINF2,0,1.000000e+00
3,chr11,57719406,57720297,chr11:57719406-57720297,chr11,57720297,57720298,GM32915,0,1.000000e+00
4,chr8,119436687,119437532,chr8:119436687-119437532,chr8,119437532,119437533,GM63757,0,1.000000e+00
...,...,...,...,...,...,...,...,...,...,...
50488308,chr1,24295262,24296008,chr1:24295262-24296008,chr1,23296008,23296009,GM53541,1000000,4.248354e-18
50488309,chr4,134886177,134887106,chr4:134886177-134887106,chr4,133887106,133887107,GM68734,1000000,4.248354e-18
50488310,chr5,151175820,151176702,chr5:151175820-151176702,chr5,150176702,150176703,GM72600,1000000,4.248354e-18
50488311,chr12,82274207,82275049,chr12:82274207-82275049,chr12,81275049,81275050,GM3693,1000000,4.248354e-18


## Chromosome-Specific Data Formatting


In [8]:
total_TG_pseudobulk_global, pseudobulk_chrom_dict = tdf.aggregate_pseudobulk_datasets(force_recalculate=False)
total_TG_pseudobulk_global.head()

  - Found existing global and per-chrom pseudobulk; loading...


,AAACAGCCAAACTCAT-1,AAACATGCAACCTGGT-1,AAACATGCAATGAATG-1,AAACATGCAATTATGC-1,AAACATGCACCTATAG-1,AAACATGCAGGATAAC-1,AAACCAACATAATCGT-1,AAACCGAAGAGGGACT-1,AAACCGCGTGAACAAA-1,AAACCGCGTTAACGAT-1,...,TTTGTGGCAATAATGG-1,TTTGTGGCACAATGCC-1,TTTGTGGCAGCACCAT-1,TTTGTGGCAGCACGTT-1,TTTGTGGCATGAATAG-1,TTTGTGTTCTGTGAGT-1,TTTGTTGGTACAATGT-1,TTTGTTGGTCCTTCTC-1,TTTGTTGGTTAAGCCA-1,TTTGTTGGTTAATGAC-1
0610005C13RIK,-0.282479,-0.191067,-0.216796,-0.156428,-0.177397,-0.279658,0.432268,-0.227272,-0.187673,-0.250557,...,0.568745,1.896808,-0.263528,-0.264592,-0.269208,-0.222234,-0.130673,-0.277631,1.463841,-0.217445
0610009E02RIK,-0.165841,-0.165913,0.062958,-0.111208,-0.164909,-0.106381,-0.169892,-0.160091,0.923441,-0.157398,...,-0.169728,-0.149924,-0.167922,1.083216,-0.004862,-0.153275,-0.162185,1.451688,-0.107407,-0.015187
0610009L18RIK,-0.051948,0.299400,0.167516,-0.096523,-0.093400,0.071991,-0.039363,-0.081976,-0.025130,-0.056363,...,-0.094679,0.058806,-0.091815,-0.013607,-0.035489,-0.096523,0.154920,-0.094260,-0.096523,-0.028368
0610030E20RIK,-0.261493,-0.235710,0.232739,-0.293941,-0.110585,-0.017197,0.319739,0.087553,-0.078334,0.027293,...,0.339856,0.438889,0.014483,0.186608,-0.043661,-0.250167,0.187243,-0.232593,0.342870,-0.353937
0610038B21RIK,-0.107516,0.158184,-0.072866,-0.101787,-0.104079,-0.072953,-0.036954,0.012016,0.151483,0.255801,...,0.322489,-0.088230,-0.107516,-0.086293,-0.017479,-0.016538,-0.084196,0.731441,0.275296,-0.107516


In [9]:
tdf.create_chrom_files(total_TG_pseudobulk_global, pseudobulk_chrom_dict)

Saved TF tensor with 1223 TFs and 5199 metacells.
  - Number of chromosomes: 4: ['chr1', 'chr2', 'chr3', 'chr4']
  - Processing chromosomes for dataset: mESC_E7.5_rep1_muon_preprocessing_filter_tfs
  - All required output files exist for chr1. Skipping preprocessing.
  - All required output files exist for chr2. Skipping preprocessing.
  - All required output files exist for chr3. Skipping preprocessing.
  - All required output files exist for chr4. Skipping preprocessing.
Preprocessing complete. Wrote per-sample/per-chrom data for MultiomicTransformerDataset.


We can check that all of the required cached files exist for the experiment.

In [10]:
missing_files = tdf.check_cached_file_exist()
print(f"Missing {len(missing_files)} cached files")

All required files are present.


Missing 0 cached files


# Testing how the data prep and caching interacts with the experiment_loader class

In [ ]:
importlib.reload(experiment_loader)

In [12]:
import multiomic_transformer.utils.experiment_loader as experiment_loader
from multiomic_transformer.models.model_simplified import MultiomicTransformer

os.makedirs(
    f"/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/experiments/{dataset_name}/model_training_001",
    exist_ok=True
    )

exp = experiment_loader.ExperimentLoader(
    experiment_dir = "/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/experiments/",
    experiment_name=dataset_name,
    model_num=1,
)

exp.tf_names = tdf.tfs
exp.tg_names = tdf.tgs

preprocesing_setting_dict = {
    "COMMON_DATA" : tdf.file_paths["training_cache"]["common"]["dir"],
    "SAMPLE_DATA_CACHE_DIR": tdf.file_paths["training_cache"]["dataset_dir"],
    "CHROM_IDS": tdf.chrom_list,
}

exp.preprocessing_settings = preprocesing_setting_dict

print("Creating dataset...")
dataset = exp.create_multichrom_dataset(
    max_cached=100,
)


Creating dataset...


In [13]:
batch_size=128

print("Creating dataloaders...")
train_loader, val_loader, test_loader = exp.prepare_dataloader(
    dataset,
    batch_size=batch_size,
    world_size=1,
    rank=0,
    num_workers=8,
    pin_memory=True,
)

print("Creating and fitting scalers...")
scalers = exp.create_scalers(
    dataset=dataset,
    dataloader=train_loader,
    max_batches=100,
)

Creating dataloaders...
Creating and fitting scalers...


In [14]:
from scipy.stats import norm
from sklearn.metrics import roc_auc_score

def format_grn(df):
    def inverse_normal_transform(x):
        r = x.rank(method="average")
        n = len(x)
        p = (r - 0.5) / n          # avoids 0 and 1
        return norm.ppf(p)
    
    # Apply rank-based inverse normal transform (INT)
    df["Score"] = df.groupby("Source")["Score"].transform(inverse_normal_transform)
    
    df = df.dropna()
    
    df["Source"] = df["Source"].astype(str).str.upper()
    df["Target"] = df["Target"].astype(str).str.upper()
    
    return df

def quick_pooled_auroc(exp, labeled_df):
    balanced = exp._balance_pos_neg(labeled_df, random_state=42)
    y = balanced["_in_gt"].astype(int).to_numpy()
    s = balanced["Score"].to_numpy()
    
    auroc = roc_auc_score(y, s)
    
    return auroc

def quick_per_tf_auroc(exp, labeled_df):
    per_tf_auroc = []
    
    for tf, group in labeled_df.groupby("Source"):
        balanced = exp._balance_pos_neg(group, random_state=42)
        y = balanced["_in_gt"].astype(int).to_numpy()
        s = balanced["Score"].to_numpy()
        
        if len(np.unique(y)) > 1:
            auroc = roc_auc_score(y, s)
            per_tf_auroc.append(auroc)
        else:
            per_tf_auroc.append(np.nan)  # or some default value for TFs with only pos or neg examples
    
    median_per_tf_auroc = np.nanmedian(per_tf_auroc)
    
    return median_per_tf_auroc

In [15]:
first_batch = next(iter(train_loader))
atac_wins, tf_tensor, tg_tensor, bias, tf_ids, tg_ids, motif_mask = first_batch
print("Shapes of first batch:")
print(f"atac_wins: {atac_wins.shape}")
print(f"tf_tensor: {tf_tensor.shape}")
print(f"tg_tensor: {tg_tensor.shape}")
print(f"bias: {bias.shape if bias is not None else None}")
print(f"tf_ids: {tf_ids.shape}")
print(f"tg_ids: {tg_ids.shape}")

Shapes of first batch:
atac_wins: torch.Size([128, 13849, 1])
tf_tensor: torch.Size([128, 1223])
tg_tensor: torch.Size([128, 914])
bias: torch.Size([914, 13849])
tf_ids: torch.Size([1223])
tg_ids: torch.Size([914])


In [16]:
import time

epochs = 5
bias_scale = 0.0
num_layers = 1
num_heads = 1
d_model = 64
d_ff = d_model*4
window_pool_size = 128

max_training_batches = 25
max_grad_attrib_batches = 10

training_time_size_dict = {}

GROUND_TRUTH_DIR = Path(PROJECT_DIR) / "data" / "ground_truth_files"
ground_truth = exp.load_ground_truth(GROUND_TRUTH_DIR / "chipatlas_macrophage.csv")

print(f"\n=== Running experiment with window_pool_size={window_pool_size} ===")

# ----- Create a new model -----
model = exp.create_new_model(
dataset=dataset,
    bias_scale=bias_scale,
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    d_ff=d_ff,
    dropout=0.1,
    window_pool_size=window_pool_size,
)

# ----- Train the model and log GPU memory and batch times -----
train_time_start = time.time()
model = exp.train_timed(
    model,
    train_loader,
    val_loader,
    num_epochs=epochs,
    validate_every=1,
    max_batches=max_training_batches,
    monitor_gpu_memory=True,
    profile_batches=True,
)
train_time_end = time.time()

# ----- Gradient Attribution on the Trained Model -----
selected_experiment_dir = exp.model_training_dir / "grn_results_by_checkpoint" / "testing_bias_scale"
start_time = time.time()
grad_attr_df, grad_batch_dfs = exp.run_gradient_attribution(
    selected_experiment_dir=exp.model_training_dir,
    model=model,
    test_loader=test_loader,
    tf_scaler=scalers["tf_scaler"],
    tg_scaler=scalers["tg_scaler"],
    tf_names = exp.tf_names,
    tg_names = exp.tg_names,
    use_amp=False,
    max_batches=max_grad_attrib_batches,
    device=exp.device,
    save_every_n_batches=1,
    max_tgs_per_batch=128,
)

end_time = time.time()
print(f"  - Gradient attribution finished {max_grad_attrib_batches} batches in {end_time - start_time:.2f} seconds.")

# ----- Calculate the AUROC -----
ground_truth_df, gt_lookup = ground_truth

df = format_grn(grad_attr_df)[["Source", "Target", "Score"]].copy()

labeled_df = exp.create_ground_truth_comparison_df(df, gt_lookup, "ChIP-Atlas macrophage")

y_all = labeled_df["_in_gt"].fillna(0).astype(int).to_numpy()
s_all = labeled_df["Score"].to_numpy()

# Calculate the pooled AUROC and per-TF median AUROC
pooled_auroc = quick_pooled_auroc(exp, labeled_df)
per_tf_median_auroc = quick_per_tf_auroc(exp, labeled_df)
auroc_df = pd.DataFrame({
    "window_pool_size": window_pool_size,
    "pooled_auroc": pooled_auroc,
    "per_tf_median_auroc": per_tf_median_auroc,
}, index=[0])

# Calculate the training time and average time per epoch
training_time = train_time_end - train_time_start
avg_time_per_epoch = training_time / epochs
training_time_size_dict[window_pool_size] = {
    "total": training_time,
    "avg_per_epoch": avg_time_per_epoch
}

exp.gpu_mem_log_df["window_pool_size"] = window_pool_size
exp.batch_profile_df["window_pool_size"] = window_pool_size


=== Running experiment with window_pool_size=128 ===


Epoch 1/5:  12%|██████▎                                              | 3/25 [00:00<00:03,  5.95it/s]

loader_s      0.106229
transfer_s    0.002545
forward_s     0.279353
backward_s    0.111022
optim_s       0.000000
dtype: float64
loader_s      0.053283
transfer_s    0.002051
forward_s     0.144630
backward_s    0.064911
optim_s       0.000000
dtype: float64
loader_s      0.035598
transfer_s    0.002474
forward_s     0.100572
backward_s    0.060270
optim_s       0.000000
dtype: float64
loader_s      0.026777
transfer_s    0.002427
forward_s     0.077669
backward_s    0.048454
optim_s       0.005876
dtype: float64
loader_s      0.021457
transfer_s    0.002423
forward_s     0.063618
backward_s    0.041353
optim_s       0.004701
dtype: float64
loader_s      0.017906
transfer_s    0.002279
forward_s     0.054148
backward_s    0.036411
optim_s       0.003918
dtype: float64


Epoch 1/5:  44%|██████████████████████▉                             | 11/25 [00:00<00:00, 19.55it/s]

loader_s      0.015427
transfer_s    0.002426
forward_s     0.047556
backward_s    0.033278
optim_s       0.003358
dtype: float64
loader_s      0.013539
transfer_s    0.002410
forward_s     0.042682
backward_s    0.030769
optim_s       0.003131
dtype: float64
loader_s      0.012072
transfer_s    0.002410
forward_s     0.038761
backward_s    0.028791
optim_s       0.002783
dtype: float64
loader_s      0.010886
transfer_s    0.002324
forward_s     0.035560
backward_s    0.027093
optim_s       0.002505
dtype: float64
loader_s      0.009915
transfer_s    0.002413
forward_s     0.033049
backward_s    0.025940
optim_s       0.002277
dtype: float64
loader_s      0.009107
transfer_s    0.002404
forward_s     0.030910
backward_s    0.024863
optim_s       0.002213
dtype: float64
loader_s      0.008419
transfer_s    0.002404
forward_s     0.029191
backward_s    0.023966
optim_s       0.002042
dtype: float64
loader_s      0.007832
transfer_s    0.002344
forward_s     0.027589
backward_s    0.02309

...

loader_s      0.004326
transfer_s    0.002395
forward_s     0.010017
backward_s    0.014290
optim_s       0.000612
dtype: float64
loader_s      0.004292
transfer_s    0.002395
forward_s     0.009996
backward_s    0.014279
optim_s       0.000607
dtype: float64
loader_s      0.004260
transfer_s    0.002388
forward_s     0.009970
backward_s    0.014258
optim_s       0.000602
dtype: float64
loader_s      0.004227
transfer_s    0.002396
forward_s     0.009963
backward_s    0.014262
optim_s       0.000597
dtype: float64


loader_s      0.004195
transfer_s    0.002395
forward_s     0.009942
backward_s    0.014252
optim_s       0.000606
dtype: float64
loader_s      0.004164
transfer_s    0.002395
forward_s     0.009921
backward_s    0.014242
optim_s       0.000601
dtype: float64


Epoch 5/5 | Train Loss: 0.1059 | Val MSE: 0.0969 | R2 (Unscaled): -0.161 | R2 (Scaled): -0.161 | LR: 1.00e-04 | Time: 0.8s


Gradient attributions: 100%|███████████████████████████████████| 10/10 [00:11<00:00,  1.13s/batches]


  - Gradient attribution finished 10 batches in 11.64 seconds.
